In [ ]:
from dotenv import load_dotenv

# Load environment variables from a .env file for API keys and other configurations
load_dotenv(".env")

In [ ]:
# Import the InMemoryStore class for storing memories in memory (no persistence)
from langgraph.store.memory import InMemoryStore

# Initialize an in-memory store instance. This will hold our temporary data.
in_memory_store = InMemoryStore()

In [ ]:
# Define a user ID for memory storage
user_id = "1"

# Set the namespace for storing/retrieving memories as a tuple
namespace_for_memory = (user_id, "memories")

In [ ]:
import uuid

# Generate a unique ID for the memory item
memory_id = str(uuid.uuid4())

# Create a simple dictionary to represent our memory
memory = {"food_preference" : "I like pizza"}

# Save the memory in the defined namespace using the put() method
in_memory_store.put(namespace_for_memory, memory_id, memory)

In [ ]:
# Retrieve all stored memories for the given namespace
memories = in_memory_store.search(namespace_for_memory)

# View the last (most recent) memory as a dictionary
# The most important attribute here is 'value', which holds our stored data.
print(memories[-1].dict())

In [ ]:
# For short-term memory (managing checkpoints between threads)
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()

# For long-term memory (storing information across threads)
from langgraph.store.memory import InMemoryStore
in_memory_store = InMemoryStore()

# Example of compiling a graph with both components
# graph = graph.compile(checkpointer=checkpointer, store=in_memory_store)

In [ ]:
# Import necessary libraries from Pydantic and Python's typing module
from pydantic import BaseModel, Field
from typing import Literal
from typing_extensions import TypedDict

# Define a Pydantic model for our router's structured output.
class RouterSchema(BaseModel):
    """Analyze the unread email and route it according to its content."""
    
    # A field for the LLM to explain its step-by-step reasoning.
    reasoning: str = Field(description="Step-by-step reasoning behind the classification.")
    
    # A field to hold the final classification.
    # The `Literal` type restricts the output to one of these three specific strings.
    classification: Literal["ignore", "respond", "notify"] = Field(
        description="The classification of an email."
    )

In [ ]:
# Import the base state class from LangGraph
from langgraph.graph import MessagesState

# Define the central state object for our graph, inheriting from MessagesState.
# MessagesState automatically gives us a `messages` list to track conversation history.
class State(MessagesState):
    """The central state of the graph."""
    # This field will hold the initial raw email data.
    email_input: dict
    
    # This field will store the decision made by our triage router.
    classification_decision: Literal["ignore", "respond", "notify"]

In [ ]:
# Define a TypedDict for the initial input to our entire workflow.
class StateInput(TypedDict):
    """The required input shape for the graph."""
    # The workflow must be started with a dictionary containing an 'email_input' key.
    email_input: dict

In [ ]:
# Import prompts from the helper library
from email_assistant.prompts import (
    triage_system_prompt, 
    triage_user_prompt, 
    agent_system_prompt_hitl_memory, 
    default_triage_instructions, 
    default_background, 
    default_response_preferences, 
    default_cal_preferences, 
    MEMORY_UPDATE_INSTRUCTIONS, 
    MEMORY_UPDATE_INSTRUCTIONS_REINFORCEMENT
)
from email_assistant.tools.default.prompt_templates import HITL_MEMORY_TOOLS_PROMPT

# We can inspect these imported strings. For example, the default background:
print("--- Default Background ---")
print(default_background)

# And the default triage instructions:
print("\n--- Default Triage Instructions ---")
print(default_triage_instructions)

In [ ]:
from rich.markdown import Markdown

# Inspect the main agent system prompt
print("--- Agent System Prompt Template ---")
print(agent_system_prompt_hitl_memory)

# Inspect the memory update instructions
print("\n--- Memory Update Instructions ---")
Markdown(MEMORY_UPDATE_INSTRUCTIONS)

In [ ]:
# Import utility functions
from email_assistant.utils import parse_email, format_for_display, format_email_markdown

# We can inspect the HITL_MEMORY_TOOLS_PROMPT, which is a simple text description
# of the tools available to the LLM.
print("--- Tools Prompt for LLM ---")
print(HITL_MEMORY_TOOLS_PROMPT)

In [ ]:
from datetime import datetime
from langchain_nebius import ChatNebius
from langchain_core.tools import tool

# --- Agent Tools ---

# A tool for writing and sending an email.
@tool
def write_email(to: str, subject: str, content: str) -> str:
    """Write and send an email."""
    # Placeholder response - in a real app, this would send an email.
    return f"Email sent to {to} with subject '{subject}' and content: {content}"

# A tool for scheduling a calendar meeting.
@tool
def schedule_meeting(
    attendees: list[str], subject: str, duration_minutes: int, preferred_day: datetime, start_time: int
) -> str:
    """Schedule a calendar meeting."""
    # Placeholder response - in a real app, this would check the calendar and schedule.
    date_str = preferred_day.strftime("%A, %B %d, %Y")
    return f"Meeting '{subject}' scheduled on {date_str} at {start_time} for {duration_minutes} minutes with {len(attendees)} attendees"

# A tool to check calendar availability.
@tool
def check_calendar_availability(day: str) -> str:
    """Check calendar availability for a given day."""
    # Placeholder response - in a real app, this would check an actual calendar.
    return f"Available times on {day}: 9:00 AM, 2:00 PM, 4:00 PM"

# A tool to ask the user a clarifying question.
@tool
class Question(BaseModel):
      """Question to ask user."""
      content: str

# A tool to signify that the task is complete.
@tool
class Done(BaseModel):
      """E-mail has been sent."""
      done: bool
    
# A list of all tools available to the agent.
tools = [
    write_email, 
    schedule_meeting, 
    check_calendar_availability, 
    Question, 
    Done
]

# A dictionary to easily access tools by their name.
tools_by_name = {tool.name: tool for tool in tools}

# --- LLM Initialization ---

# Initialize the base LLM. We are using a Nebius model.
llm = ChatNebius(
    model="meta-llama/Llama-3.3-70B-Instruct",
    temperature=0.0,
)

# Create a specialized LLM for the router, forcing it to use our RouterSchema.
llm_router = llm.with_structured_output(RouterSchema) 

# Create a specialized LLM for the main agent, binding our tools to it.
# `tool_choice="required"` forces the LLM to call a tool at every step.
llm_with_tools = llm.bind_tools(tools, tool_choice="required")

In [ ]:
# A function to retrieve memory from the store or initialize it with defaults.
def get_memory(store, namespace, default_content=None):
    """Get memory from the store or initialize with default if it doesn't exist."""
    # Use the store's .get() method to search for an item with a specific key.
    user_preferences = store.get(namespace, "user_preferences")
    
    # If the item exists, return its value (the stored string).
    if user_preferences:
        return user_preferences.value
    
    # If the item does not exist, this is the first time we're accessing this memory.
    else:
        # Use the store's .put() method to create the memory item with default content.
        store.put(namespace, "user_preferences", default_content)
        # Return the default content to be used in this run.
        return default_content

In [ ]:
from langchain_core.messages import AIMessage

# A Pydantic model to structure the output of our memory update LLM call.
class UserPreferences(BaseModel):
    """Updated user preferences based on user's feedback."""
    # A field for the LLM to explain its reasoning, useful for debugging.
    chain_of_thought: str = Field(description="Reasoning about which user preferences need to add / update if required")
    # The final, updated string of user preferences.
    user_preferences: str = Field(description="Updated user preferences")

# This function intelligently updates the memory store based on user feedback.
def update_memory(store, namespace, messages):
    """Update memory profile in the store."""
    # First, get the current memory from the store to provide it as context.
    user_preferences = store.get(namespace, "user_preferences")
    
    # Initialize a new LLM instance for this task, configured for structured output.
    memory_updater_llm = llm.with_structured_output(UserPreferences)
    
    # Filter out any previous AI messages with tool calls.
    # Passing these complex objects can sometimes cause errors in the downstream LLM call.
    messages_to_send = [
        msg for msg in messages
        if not (isinstance(msg, AIMessage) and hasattr(msg, 'tool_calls') and msg.tool_calls)
    ]
    
    # Invoke the LLM with the memory prompt, current preferences, and user feedback.
    result = memory_updater_llm.invoke(
        [
            # The system prompt that instructs the LLM on how to update memory.
            {"role": "system", "content": MEMORY_UPDATE_INSTRUCTIONS.format(current_profile=user_preferences.value, namespace=namespace)},
        ] + messages_to_send
    )
    
    # Save the newly generated preference string back into the store.
    store.put(namespace, "user_preferences", result.user_preferences)

In [ ]:
from langgraph.store.base import BaseStore
from langgraph.types import Command, END

# Define the first node in our graph, the triage router.
def triage_router(state: State, store: BaseStore) -> Command:
    """Analyze email content to decide the next step."""
    # Unpack the raw email data using our utility function.
    author, to, subject, email_thread = parse_email(state["email_input"])
    
    # Format the email content into a clean string for the LLM.
    user_prompt = triage_user_prompt.format(
        author=author, to=to, subject=subject, email_thread=email_thread
    )
    email_markdown = format_email_markdown(subject, author, to, email_thread)

    # MEMORY INTEGRATION: Fetch the latest triage instructions.
    # If they don't exist, it will use the `default_triage_instructions`.
    triage_instructions = get_memory(store, ("email_assistant", "triage_preferences"), default_triage_instructions)

    # Format the system prompt, injecting the retrieved triage instructions.
    system_prompt = triage_system_prompt.format(
        background=default_background,
        triage_instructions=triage_instructions,
    )

    # Invoke the LLM router, which is configured to return our `RouterSchema`.
    result = llm_router.invoke(
        [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
    )

    # Based on the LLM's classification, decide which node to go to next.
    classification = result.classification
    if classification == "respond":
        print("📧 Classification: RESPOND - This email requires a response")
        goto = "response_agent"
        update = {
            "classification_decision": result.classification,
            "messages": [{"role": "user", "content": f"Respond to the email: {email_markdown}"}],
        }
    elif classification == "ignore":
        print("🚫 Classification: IGNORE - This email can be safely ignored")
        goto = END
        update = {"classification_decision": classification}
    elif classification == "notify":
        print("🔔 Classification: NOTIFY - This email contains important information")
        goto = "triage_interrupt_handler"
        update = {"classification_decision": classification}
    else:
        raise ValueError(f"Invalid classification: {classification}")
    
    # Return a Command object to tell LangGraph where to go next and what to update.
    return Command(goto=goto, update=update)

In [ ]:
# This is the primary reasoning node for the response agent.
def llm_call(state: State, store: BaseStore):
    """LLM decides whether to call a tool or not, using stored preferences."""

    # Fetch the user's latest calendar preferences from the memory store.
    cal_preferences = get_memory(store, ("email_assistant", "cal_preferences"), default_cal_preferences)
    
    # Fetch the user's latest response (writing style) preferences.
    response_preferences = get_memory(store, ("email_assistant", "response_preferences"), default_response_preferences)

    # Filter out previous AI messages with tool calls to prevent API errors.
    messages_to_send = [
        msg for msg in state["messages"]
        if not (isinstance(msg, AIMessage) and hasattr(msg, 'tool_calls') and msg.tool_calls)
    ]

    # Invoke the main LLM, which is bound to our set of tools.
    # The prompt is formatted with the preferences retrieved from memory.
    response = llm_with_tools.invoke(
        [
            {"role": "system", "content": agent_system_prompt_hitl_memory.format(
                tools_prompt=HITL_MEMORY_TOOLS_PROMPT,
                background=default_background,
                response_preferences=response_preferences, 
                cal_preferences=cal_preferences)}
        ]
        + messages_to_send
    )
    
    # Return the LLM's response to be added to the state.
    return {"messages": [response]}

In [ ]:
from langgraph.types import interrupt

# Define the interrupt handler for the triage step.
def triage_interrupt_handler(state: State, store: BaseStore) -> Command:
    """Handles interrupts from the triage step, pausing for user input."""
    # Parse the email input to format it for display.
    author, to, subject, email_thread = parse_email(state["email_input"])
    email_markdown = format_email_markdown(subject, author, to, email_thread)
    
    # This data structure defines the interrupt, specifying the action, allowed responses, and content.
    request = {
        "action_request": {"action": f"Email Assistant: {state['classification_decision']}", "args": {}},
        "config": {"allow_ignore": True, "allow_respond": True, "allow_edit": False, "allow_accept": False},
        "description": email_markdown,
    }
    
    # The `interrupt()` function pauses the graph and waits for a response.
    response = interrupt([request])[0]
    
    # Process the user's response.
    if response["type"] == "response":
        # The user decided to respond, overriding the 'notify' classification.
        user_input = response["args"]
        messages_for_memory = [{
            "role": "user",
            "content": f"The user decided to respond to the email, so update the triage preferences to capture this."
        }]
        # Call `update_memory` to teach the agent this new preference.
        update_memory(store, ("email_assistant", "triage_preferences"), messages_for_memory)
        goto = "response_agent"
        update = {"messages": [{"role": "user", "content": f"User wants to reply. Use this feedback: {user_input}"}]}
    elif response["type"] == "ignore":
        # The user confirmed the email should be ignored.
        messages_for_memory = [{
            "role": "user",
            "content": f"The user decided to ignore the email even though it was classified as notify. Update triage preferences to capture this."
        }]
        # Update memory to reinforce this preference.
        update_memory(store, ("email_assistant", "triage_preferences"), messages_for_memory)
        goto = END
        update = {}
    else:
        raise ValueError(f"Invalid response: {response}")
    
    # Return a Command to direct the graph's next step.
    return Command(goto=goto, update=update)

In [ ]:
# The main interrupt handler for reviewing tool calls.
def interrupt_handler(state: State, store: BaseStore) -> Command:
    """Creates an interrupt for human review of tool calls and updates memory."""
    # We'll build up a list of new messages to add to the state.
    result = []
    # By default, we'll loop back to the LLM after this.
    goto = "llm_call"

    # The agent can propose multiple tool calls, so we loop through them.
    for tool_call in state["messages"][-1].tool_calls:
        # We only want to interrupt for certain "high-stakes" tools.
        hitl_tools = ["write_email", "schedule_meeting", "Question"]
        if tool_call["name"] not in hitl_tools:
            # For other tools (like check_calendar), execute them without interruption.
            tool = tools_by_name[tool_call["name"]]
            observation = tool.invoke(tool_call["args"])
            result.append({"role": "tool", "content": observation, "tool_call_id": tool_call["id"]})
            continue
            
        # Format the proposed action for display to the human reviewer.
        email_input = state["email_input"]
        author, to, subject, email_thread = parse_email(email_input)
        original_email_markdown = format_email_markdown(subject, author, to, email_thread)
        tool_display = format_for_display(tool_call)
        description = original_email_markdown + tool_display

        # Configure allowed actions in the HITL interface based on the tool.
        config = {}
        if tool_call["name"] in ["write_email", "schedule_meeting"]:
            config = {"allow_ignore": True, "allow_respond": True, "allow_edit": True, "allow_accept": True}
        elif tool_call["name"] == "Question":
            config = {"allow_ignore": True, "allow_respond": True, "allow_edit": False, "allow_accept": False}
        else:
            raise ValueError(f"Invalid tool call: {tool_call['name']}")

        # Create and send the interrupt request.
        request = {"action_request": {"action": tool_call["name"], "args": tool_call["args"]}, "config": config, "description": description}
        response = interrupt([request])[0]

        # --- MEMORY UPDATE LOGIC BASED ON USER RESPONSE ---
        if response["type"] == "accept":
            # User approved. No memory update needed. Execute the tool.
            tool = tools_by_name[tool_call["name"]]
            observation = tool.invoke(tool_call["args"])
            result.append({"role": "tool", "content": observation, "tool_call_id": tool_call["id"]})
                        
        elif response["type"] == "edit":
            # User directly edited the action. This is direct feedback.
            initial_tool_call = tool_call["args"]
            edited_args = response["args"]["args"]
            
            # Update memory based on the specific tool that was edited.
            if tool_call["name"] == "write_email":
                update_memory(store, ("email_assistant", "response_preferences"), [{
                    "role": "user",
                    "content": f"User edited the email. Initial draft: {initial_tool_call}. Edited draft: {edited_args}. {MEMORY_UPDATE_INSTRUCTIONS_REINFORCEMENT}"
                }])
            elif tool_call["name"] == "schedule_meeting":
                update_memory(store, ("email_assistant", "cal_preferences"), [{
                    "role": "user",
                    "content": f"User edited the meeting. Initial invite: {initial_tool_call}. Edited invite: {edited_args}. {MEMORY_UPDATE_INSTRUCTIONS_REINFORCEMENT}"
                }])
                
            # Execute the tool with the user's edited arguments.
            tool = tools_by_name[tool_call["name"]]
            observation = tool.invoke(edited_args)
            result.append({"role": "tool", "content": observation, "tool_call_id": tool_call["id"]})

        elif response["type"] == "ignore":
            # User decided this action shouldn't be taken. This is triage feedback.
            update_memory(store, ("email_assistant", "triage_preferences"), state["messages"] + [{
                "role": "user",
                "content": f"User ignored the proposal to {tool_call['name']}. This email should not have been classified as 'respond'. {MEMORY_UPDATE_INSTRUCTIONS_REINFORCEMENT}"
            }])
            result.append({"role": "tool", "content": "User ignored this. End the workflow.", "tool_call_id": tool_call["id"]})
            goto = END

        elif response["type"] == "response":
            # User gave natural language feedback.
            user_feedback = response["args"]
            
            # Capture this feedback and use it to update memory.
            if tool_call["name"] == "write_email":
                update_memory(store, ("email_assistant", "response_preferences"), state["messages"] + [{
                    "role": "user",
                    "content": f"User gave feedback on the email draft: {user_feedback}. {MEMORY_UPDATE_INSTRUCTIONS_REINFORCEMENT}"
                }])
            elif tool_call["name"] == "schedule_meeting":
                update_memory(store, ("email_assistant", "cal_preferences"), state["messages"] + [{
                    "role": "user",
                    "content": f"User gave feedback on the meeting invite: {user_feedback}. {MEMORY_UPDATE_INSTRUCTIONS_REINFORCEMENT}"
                }])
            
            # Don't execute the tool. Pass the feedback back to the agent for another attempt.
            result.append({"role": "tool", "content": f"User gave feedback: {user_feedback}", "tool_call_id": tool_call["id"]})
    
    # Return a command with the next node and the messages to add to the state.
    return Command(goto=goto, update={"messages": result})

In [ ]:
# This function determines the next step after the LLM has made its decision.
def should_continue(state: State) -> Literal["interrupt_handler", "__end__"]:
    """Route to the interrupt handler or end the workflow if the 'Done' tool is called."""
    # Get the most recent message, which contains the agent's proposed action.
    last_message = state["messages"][-1]
    
    # Check if the last message contains any tool calls.
    if last_message.tool_calls:
        # Loop through each proposed tool call.
        for tool_call in last_message.tool_calls: 
            # If the agent has decided it's finished, we end the workflow.
            if tool_call["name"] == "Done":
                return END
            # For any other tool, we proceed to the human review step.
            else:
                return "interrupt_handler"
    # This path should not be reached if tool_choice="required" is set on the LLM
    return END

In [ ]:
from langgraph.graph import StateGraph, START
from email_assistant.utils import show_graph

# --- Part 1: Build the Response Agent Subgraph ---
agent_builder = StateGraph(State)
agent_builder.add_node("llm_call", llm_call)
agent_builder.add_node("interrupt_handler", interrupt_handler)
agent_builder.add_edge(START, "llm_call")
agent_builder.add_conditional_edges(
    "llm_call",
    should_continue,
    {
        "interrupt_handler": "interrupt_handler",
        END: END,
    },
)
agent_builder.add_edge("interrupt_handler", "llm_call")
response_agent = agent_builder.compile()

# --- Part 2: Build the Overall Workflow ---
overall_workflow = (
    StateGraph(State, input=StateInput)
    .add_node("triage_router", triage_router)
    .add_node("triage_interrupt_handler", triage_interrupt_handler)
    .add_node("response_agent", response_agent)
    .add_edge(START, "triage_router")
    # Edges from the triage node are determined by its returned Command object
)

# Compile the final, complete graph.
email_assistant = overall_workflow.compile()

# Visualize the graph structure.
show_graph(email_assistant)

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# Define a helper function to display the content of our memory store.
def display_memory_content(store, namespace=None):
    """A utility to print the current state of the memory store."""
    # Print a header for clarity.
    print("\n======= CURRENT MEMORY CONTENT =======")
    
    # If a specific namespace is requested, show only that one.
    if namespace:
        memory = store.get(namespace, "user_preferences")
        print(f"\n--- {namespace[1]} ---")
        if memory:
            print(memory.value)
        else:
            print("No memory found")
            
    # If no specific namespace is given, show all of them.
    else:
        for ns in [
            ("email_assistant", "triage_preferences"),
            ("email_assistant", "response_preferences"),
            ("email_assistant", "cal_preferences"),
            ("email_assistant", "background")
        ]:
            memory = store.get(ns, "user_preferences")
            print(f"\n--- {ns[1]} ---")
            if memory:
                print(memory.value)
            else:
                print("No memory found")
            print("=======================================\n")

In [ ]:
# Define the input email for our test case.
email_input_respond = {
    "to": "Lance Martin <lance@company.com>",
    "author": "Project Manager <pm@client.com>",
    "subject": "Tax season let's schedule call",
    "email_thread": "Lance,\n\nIt's tax season again, and I wanted to schedule a call to discuss your tax planning strategies for this year. I have some suggestions that could potentially save you money.\n\nAre you available sometime next week? Tuesday or Thursday afternoon would work best for me, for about 45 minutes.\n\nRegards,\nProject Manager"
}

# --- Setup for a new test run ---
# Initialize a new checkpointer and a fresh, empty memory store.
checkpointer = MemorySaver()
store = InMemoryStore()
# Compile our graph, connecting it to our new checkpointer and store.
graph = email_assistant.compile(checkpointer=checkpointer, store=store)

# Create a unique ID and configuration for this conversation thread.
thread_id_1 = uuid.uuid4()
thread_config_1 = {"configurable": {"thread_id": thread_id_1}}

# Run the graph until its first interrupt.
print("Running the graph until the first interrupt...")
for chunk in graph.stream({"email_input": email_input_respond}, config=thread_config_1):
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

# Check the memory state after the first interrupt. 
# We expect to see the default preferences loaded for the first time.
display_memory_content(store)

In [ ]:
# Resume the graph by sending an 'accept' command.
print(f"\nSimulating user accepting the {Interrupt_Object.value[0]['action_request']['action']} tool call...")
for chunk in graph.stream(None, config=thread_config_1, resume_tasks=Interrupt_Object):
    # Let the graph run until its next natural pause point.
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

In [ ]:
# Resume the graph one last time with another 'accept' command.
print(f"\nSimulating user accepting the {Interrupt_Object.value[0]['action_request']['action']} tool call...")
for chunk in graph.stream(None, config=thread_config_1, resume_tasks=Interrupt_Object):
    # This should complete the workflow.
    if 'response_agent' in chunk and chunk['response_agent']['messages']:
        print("\nFinal agent messages:")
        chunk['response_agent']['messages'][-1].pretty_print()

# Check the final state of all memory namespaces.
# We expect no changes from the initial default state.
display_memory_content(store)

In [ ]:
# Get the final state of the thread and print all messages.
state = graph.get_state(thread_config_1)
for m in state.values['messages']:
    m.pretty_print()

In [ ]:
# --- Setup for a new edit test run ---
checkpointer = MemorySaver()
store = InMemoryStore()
graph = email_assistant.compile(checkpointer=checkpointer, store=store)

thread_id_2 = uuid.uuid4()
thread_config_2 = {"configurable": {"thread_id": thread_id_2}}

# Run the graph until the first interrupt.
print("Running the graph until the first interrupt...")
for chunk in graph.stream({"email_input": email_input_respond}, config=thread_config_2):
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

# Check the initial memory state for calendar preferences.
display_memory_content(store,("email_assistant", "cal_preferences"))

In [ ]:
# Define the user's edits to the proposed `schedule_meeting` tool call.
edited_schedule_args = {
    "attendees": ["pm@client.com", "lance@company.com"],
    "subject": "Tax Planning Discussion", # Changed from "Tax Planning Strategies"
    "duration_minutes": 30,             # Changed from 45 to 30
    "preferred_day": "2025-04-22",
    "start_time": 14 
}

# Create the resume payload with the 'edit' command.
resume_payload = [{"type": "edit", "args": {"args": edited_schedule_args}}]

# Resume the graph by sending the edit command.
print("\nSimulating user editing the schedule_meeting tool call...")
for chunk in graph.stream(None, config=thread_config_2, resume_tasks=[(Interrupt_Object, resume_payload)]):
    if '__interrupt__' in chunk: # Capture the next interrupt
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT (Second Interrupt):")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

# Check the memory AGAIN, after the edit has been processed.
# We expect `cal_preferences` to be updated.
print("\nChecking memory after editing schedule_meeting:")
display_memory_content(store,("email_assistant", "cal_preferences"))

In [ ]:
# The graph is paused. Let's define our edits for the email draft.
edited_email_args = {
    "to": "pm@client.com",
    "subject": "Re: Tax Planning Discussion",
    "content": "Thanks for reaching out. Sounds good. I've scheduled a 30-minute call for us next Tuesday. Looking forward to it!\n\nBest,\nLance"
}

# Create the resume payload with the 'edit' command.
resume_payload = [{"type": "edit", "args": {"args": edited_email_args}}]

# Resume the graph with the edit command for the write_email tool.
print("\nSimulating user editing the write_email tool call...")
for chunk in graph.stream(None, config=thread_config_2, resume_tasks=[(Interrupt_Object, resume_payload)]):
    if 'response_agent' in chunk and chunk['response_agent']['messages']:
        print("\nFinal agent messages:")
        chunk['response_agent']['messages'][-1].pretty_print()

# Check the 'response_preferences' memory to see what was learned.
print("\nChecking memory after editing write_email:")
display_memory_content(store, ("email_assistant", "response_preferences"))

print("\n--- Workflow Complete ---")

In [ ]:
# Get the final state of the thread and print all messages.
state = graph.get_state(thread_config_2)
for m in state.values['messages']:
    m.pretty_print()

In [ ]:
# --- Setup for a new feedback test run ---
checkpointer = MemorySaver()
store = InMemoryStore()
graph = email_assistant.compile(checkpointer=checkpointer, store=store)

thread_id_3 = uuid.uuid4()
thread_config_3 = {"configurable": {"thread_id": thread_id_3}}

# Run the graph until the first interrupt.
print("Running the graph until the first interrupt...")
for chunk in graph.stream({"email_input": email_input_respond}, config=thread_config_3):
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

# Check initial memory.
display_memory_content(store, ("email_assistant", "cal_preferences"))

In [ ]:
# Define the feedback and create the resume payload.
feedback = "Please schedule this for 30 minutes instead of 45 minutes, and I prefer afternoon meetings after 2pm."
resume_payload = [{"type": "response", "args": feedback}]

# Resume the graph with the feedback.
print(f"\nSimulating user providing feedback for the {Interrupt_Object.value[0]['action_request']['action']} tool call...")
for chunk in graph.stream(None, config=thread_config_3, resume_tasks=[(Interrupt_Object, resume_payload)]):
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

# Check memory after providing feedback.
print("\nChecking memory after providing feedback for schedule_meeting:")
display_memory_content(store, ("email_assistant", "cal_preferences"))

In [ ]:
# Accept the agent's revised proposal.
resume_payload = [{"type": "accept"}]

print(f"\nSimulating user accepting the revised {Interrupt_Object.value[0]['action_request']['action']} tool call...")
for chunk in graph.stream(None, config=thread_config_3, resume_tasks=[(Interrupt_Object, resume_payload)]):
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

# Check memory before the next feedback step.
print("\nChecking memory before next feedback:")
display_memory_content(store, ("email_assistant", "response_preferences"))

In [ ]:
# Define the feedback for the email draft.
feedback_email = "Shorter and less formal. Include a closing statement about looking forward to the meeting!"
resume_payload = [{"type": "response", "args": feedback_email}]

# Resume the graph.
print(f"\nSimulating user providing feedback for the {Interrupt_Object.value[0]['action_request']['action']} tool call...")
for chunk in graph.stream(None, config=thread_config_3, resume_tasks=[(Interrupt_Object, resume_payload)]):
    if '__interrupt__' in chunk:
        Interrupt_Object = chunk['__interrupt__'][0]
        print("\nINTERRUPT OBJECT:")
        print(f"Action Request: {Interrupt_Object.value[0]['action_request']}")

# Check memory after providing feedback for write_email.
print("\nChecking memory after providing feedback for write_email:")
display_memory_content(store, ("email_assistant", "response_preferences"))

In [ ]:
# Final acceptance.
resume_payload = [{"type": "accept"}]

print(f"\nSimulating user accepting the final {Interrupt_Object.value[0]['action_request']['action']} tool call...")
for chunk in graph.stream(None, config=thread_config_3, resume_tasks=[(Interrupt_Object, resume_payload)]):
    if 'response_agent' in chunk and chunk['response_agent']['messages']:
        print("\nFinal agent messages:")
        chunk['response_agent']['messages'][-1].pretty_print()

# Final check of memory.
print("\nChecking final memory state:")
display_memory_content(store, ("email_assistant", "response_preferences"))

In [ ]:
state = graph.get_state(thread_config_3)
for m in state.values['messages']:
    m.pretty_print()